# Progetto: Reasoning e RAG con un modello LLM

In questo notebook metto insieme una piccola pipeline con quantizzazione, prompt engineering, retrieval, generazione, sentiment analysis zero-shot e output JSON.

Per evitare problemi di accesso ai modelli gated uso di default **Qwen2.5-1.5B-Instruct**. La struttura rimane compatibile con un modello Llama 3 instruction cambiando semplicemente `MODEL_ID`.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.1 MB/s eta 0:00:00


In [2]:
import json
import re
import time
from pathlib import Path

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from threading import Thread

print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponibile: True
GPU: Tesla T4


## 1. Caricamento del modello

Uso la quantizzazione a 4 bit solo quando è disponibile CUDA. In caso contrario il notebook prova comunque a funzionare con un dtype più leggero.

In [3]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

use_4bit = torch.cuda.is_available()

bnb_config = None
if use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

print("Caricamento tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Caricamento modello...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=None if use_4bit else torch.float32,
    device_map="auto",
)

print("Modello caricato correttamente")

Caricamento tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modello caricato correttamente


## 2. Knowledge base, chunking ed embedding

Per l'esercitazione uso un documento piccolo. Lo divido in paragrafi, genero gli embedding e costruisco un indice FAISS.

In [4]:
knowledge_text = """
L'intelligenza artificiale generativa usa modelli probabilistici che stimano quale token sia più probabile dopo quelli già presenti nella sequenza.

Il Retrieval-Augmented Generation, o RAG, permette di recuperare informazioni da documenti esterni prima della generazione della risposta. In questo modo il modello dispone di un contesto più pertinente e può ridurre alcune allucinazioni.

La quantizzazione a 4 bit consente di ridurre sensibilmente la memoria richiesta da un modello linguistico. È utile soprattutto quando si lavora con GPU consumer o ambienti come Google Colab.

Il Chain of Thought è una tecnica di prompting che invita il modello a scomporre un problema complesso in più passaggi logici prima di fornire la risposta finale.

FAISS è una libreria usata per effettuare ricerche veloci tra vettori numerici. In una pipeline RAG può essere usata per trovare i chunk semanticamente più vicini alla domanda dell'utente.

I modelli Sentence Transformers trasformano frasi e documenti in vettori numerici. Testi con significato simile tendono ad avere rappresentazioni vicine nello spazio vettoriale.
""".strip()

chunks = [c.strip() for c in knowledge_text.split("\n\n") if c.strip()]
print("Numero di chunk:", len(chunks))
for i, chunk in enumerate(chunks):
    print(i, chunk[:90] + ("..." if len(chunk) > 90 else ""))

Numero di chunk: 6
0 L'intelligenza artificiale generativa usa modelli probabilistici che stimano quale token s...
1 Il Retrieval-Augmented Generation, o RAG, permette di recuperare informazioni da documenti...
2 La quantizzazione a 4 bit consente di ridurre sensibilmente la memoria richiesta da un mod...
3 Il Chain of Thought è una tecnica di prompting che invita il modello a scomporre un proble...
4 FAISS è una libreria usata per effettuare ricerche veloci tra vettori numerici. In una pip...
5 I modelli Sentence Transformers trasformano frasi e documenti in vettori numerici. Testi c...


In [5]:
embed_device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=embed_device)

doc_embeddings = embedder.encode(chunks, normalize_embeddings=True)
doc_embeddings = np.asarray(doc_embeddings, dtype="float32")

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print("Indice FAISS pronto. Vettori indicizzati:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indice FAISS pronto. Vettori indicizzati: 6


In [6]:
def retrieve_context(query, k=2):
    query_vec = embedder.encode([query], normalize_embeddings=True)
    query_vec = np.asarray(query_vec, dtype="float32")
    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "text": chunks[idx]
        })
    return results

query_test = "Come posso ridurre la memoria necessaria per eseguire un LLM?"
retrieve_context(query_test)

[{'score': 0.40954387187957764,
  'text': 'Il Chain of Thought è una tecnica di prompting che invita il modello a scomporre un problema complesso in più passaggi logici prima di fornire la risposta finale.'},
 {'score': 0.3971630334854126,
  'text': "FAISS è una libreria usata per effettuare ricerche veloci tra vettori numerici. In una pipeline RAG può essere usata per trovare i chunk semanticamente più vicini alla domanda dell'utente."}]

## 3. Prompt engineering, reasoning e RAG

Uso delimitatori chiari tra contesto e domanda. Per il reasoning chiedo al modello di ragionare internamente e di restituire una spiegazione sintetica dei passaggi utili, senza rendere il prompt troppo complesso.

In [7]:
SYSTEM_PROMPT = """
Sei un assistente esperto di intelligenza artificiale.
Rispondi in italiano in modo chiaro e sintetico.
Usa il contesto fornito quando è pertinente.
Se il problema richiede ragionamento, analizzalo con ordine e poi fornisci una spiegazione breve dei passaggi principali.
Se il contesto non contiene la risposta, dichiaralo.
""".strip()


def generate_response(user_query, use_rag=True, temperature=0.5, top_p=0.9, max_new_tokens=250):
    context = ""
    if use_rag:
        retrieved = retrieve_context(user_query, k=2)
        context = "\n\n".join(item["text"] for item in retrieved)

    user_content = f"""
<CONTESTO>
{context if context else "Nessun contesto esterno."}
</CONTESTO>

<DOMANDA>
{user_query}
</DOMANDA>
""".strip()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [8]:
q1 = "Perché la quantizzazione a 4 bit è utile quando uso una GPU con poca memoria?"
print(generate_response(q1))

La quantizzazione a 4 bit rende le informazioni contenute nei dati meno complesse da rappresentare. Quando utilizziamo una GPU con poco spazio di memoria (come quelle consumate dalle tue chiacchiere), questo sistema di rappresentazione può ridurre significativamente l'uso della memoria disponibile. Questo perché i 4 bit che rappresentano i valori possono essere interpretati come 16 possibili stati, quindi una quantità molto maggiore di questi stati può essere gestita nella stessa quantità di memoria del singolo bit. Questo sistema di rappresentazione può quindi permettere di eseguire operazioni di ricerca veloci e efficaci su vettori di grandi dimensioni, senza necessariamente imporvi una grande quantità di memoria.


In [9]:
q2 = "Se un modello usa 16 GB in precisione standard e la memoria viene ridotta circa di 4 volte, quanta memoria servirebbe approssimativamente?"
print(generate_response(q2, use_rag=False))

Per risolvere questa domanda, possiamo utilizzare i concetti di fattoria logaritmica (log) per capire quanto diminuisce la memoria a mano.

La formula da usare è:

\[ \text{Memoria finale} = \frac{\text{Memoria iniziale}}{\text{Reduttore}} \]

Dove:
- Memoria iniziale = 16 GB
- Reduttore = 4

Siccome la memoria iniziale è 16 GB e la reduttore è 4, possiamo applicare la seguente operazione:

\[ \text{Memoria finale} = \frac{16}{4} = 4 \]

Quindi, dopo aver ridotto la memoria a mano, dovremmo avere approssimativamente 4 GB di memoria disponibile.


## 4. Sentiment analysis zero-shot con output JSON

Non addestro nessun classificatore: chiedo direttamente al modello di assegnare una delle due etichette previste e di restituire solo JSON.

In [10]:
def analyze_sentiment_json(text):
    messages = [
        {
            "role": "system",
            "content": "Sei un classificatore di sentiment. Rispondi esclusivamente con JSON valido, senza markdown."
        },
        {
            "role": "user",
            "content": (
                "Classifica la recensione come positivo o negativo. "
                "Restituisci esattamente i campi sentiment e score. "
                "Lo score deve essere un numero tra 0 e 1.\n\n"
                f"Recensione: {text}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()

    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        return {"error": "JSON non trovato", "raw_response": raw}

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"error": "JSON non valido", "raw_response": raw}

In [11]:
reviews = [
    "Il modulo sugli LLM è stato intenso ma molto interessante e utile.",
    "L'esercitazione è stata confusa e non mi è servita praticamente a nulla."
]

for review in reviews:
    print("Recensione:", review)
    print(analyze_sentiment_json(review))
    print()

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Recensione: Il modulo sugli LLM è stato intenso ma molto interessante e utile.
{'sentiment': 'positive', 'score': 0.95}

Recensione: L'esercitazione è stata confusa e non mi è servita praticamente a nulla.
{'sentiment': 'negative', 'score': 0.857}



## 5. Streaming dell'output

Per simulare una risposta più fluida uso `TextIteratorStreamer` di Transformers.

In [12]:
def stream_response(user_query):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=120,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for piece in streamer:
        print(piece, end="", flush=True)

    thread.join()
    print()

stream_response("Spiega in poche righe che cosa fa un sistema RAG.")

Un sistema RAG (Recurrent Attention Graph) è un tipo di modello di rappresentazione avanzato per le informazioni a lungo termine. Il suo nome deriva dal fatto che utilizza attori ricorrenti e una struttura di grafici attenti. Questo sistema permette di ricostruire i dati nel tempo e di integrare sia l'informazione di input precedenti che di output successivi, creando un approccio più efficiente per la modellizzazione delle sequenze temporali o di grandi quantità di dati. È particolar


## Conclusione

La pipeline finale combina più elementi richiesti dalla traccia: caricamento del modello, quantizzazione, prompt engineering, reasoning, retrieval con FAISS, generazione aumentata dal contesto, sentiment analysis zero-shot, output JSON e streaming.

La soluzione è volutamente semplice perché l'obiettivo dell'esercitazione è mostrare il funzionamento dei singoli componenti senza aggiungere complessità non necessaria.